# Hypothetical Resume Embedding (HYRE) — ConFit v2

This notebook:
1. Converts your `resume-job-description-fit` data into ConFit v2 format
2. Generates hypothetical resumes via Qwen2.5-32B-Instruct
3. Produces the final job CSV with HYRE for training

Label mapping: `Good Fit` → 1, `No Fit` / `Potential Fit` → 0

## 0. Setup

In [1]:
!git clone https://github.com/tysjosh/Confit_v2.git /content/ConFit-v2

REPO_ROOT = "/content/ConFit-v2"

Cloning into '/content/ConFit-v2'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 155 (delta 45), reused 148 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 1.81 MiB | 21.01 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [2]:
# ── Install dependencies ───────────────────────────────────────────────
!pip install -q deepspeed pytorch-lightning torchmetrics wandb transformers \
    jieba nltk pandas scikit-learn tqdm xgboost faiss-gpu-cu12 rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.2 MB/s eta 0:00:0000:010:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 41.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.8 MB/s eta 0:00:00


In [3]:
import os, json, random, re, hashlib
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

In [8]:
# ── Mount Google Drive for persistent storage ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ───────────────────────────────────────────────────────────────
# Input data — pre-defined splits from Google Drive
SPLITS_DIR = "/content/drive/MyDrive/data/data_splits_v7"

# Output — stored in Drive so everything persists across runtime restarts
OUTPUT_DIR = "/content/drive/MyDrive/resume_job_fit"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Splits directory: {SPLITS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Splits directory: /content/drive/MyDrive/data/data_splits_v7
Output directory: /content/drive/MyDrive/resume_job_fit


## 1. Convert data_splits_v7 to ConFit v2 format

Uses the pre-defined train/validation/test splits from `data_splits_v7/`.  
Generates deterministic SHA-256 IDs from text content so records match across notebooks.

In [9]:
# ── Load all splits from data_splits_v7 ───────────────────────────────
def load_split(path):
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    return records

train_records = load_split(os.path.join(SPLITS_DIR, "train.jsonl"))
valid_records = load_split(os.path.join(SPLITS_DIR, "validation.jsonl"))
test_records  = load_split(os.path.join(SPLITS_DIR, "test.jsonl"))

print(f"Train: {len(train_records):,}  Valid: {len(valid_records):,}  Test: {len(test_records):,}")
print(f"Total: {len(train_records)+len(valid_records)+len(test_records):,} records")

Train: 6,400  Valid: 800  Test: 800
Total: 8,000 records


In [10]:
def text_to_id(text: str) -> str:
    """Deterministic ID from text content via SHA-256."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def records_to_df(records):
    """Convert data_splits_v7 records to a flat DataFrame."""
    rows = []
    for rec in records:
        resume_text = rec["resume"]["experience"][0]["description"]
        job_text = rec["job"]["description"]
        rows.append({
            "user_id": text_to_id(resume_text),
            "jd_no": text_to_id(job_text),
            "satisfied": int(rec["label"]),
            "resume_text": resume_text,
            "job_text": job_text,
        })
    return pd.DataFrame(rows)

train_df = records_to_df(train_records)
valid_df = records_to_df(valid_records)
test_df  = records_to_df(test_records)
all_df   = pd.concat([train_df, valid_df, test_df], ignore_index=True)

print(f"Train: {len(train_df):,} ({(train_df['satisfied']==1).sum():,} pos)")
print(f"Valid: {len(valid_df):,} ({(valid_df['satisfied']==1).sum():,} pos)")
print(f"Test:  {len(test_df):,} ({(test_df['satisfied']==1).sum():,} pos)")
print(f"\nUnique resumes: {all_df['user_id'].nunique()}")
print(f"Unique jobs:    {all_df['jd_no'].nunique()}")

Train: 6,400 (1,600 pos)
Valid: 800 (200 pos)
Test:  800 (200 pos)

Unique resumes: 643
Unique jobs:    351


In [11]:
# ── Deduplicated resume & job CSVs ───────────────────────────────────
resume_df = (
    all_df[["user_id", "resume_text"]]
    .drop_duplicates(subset="user_id")
    .rename(columns={"resume_text": "text_resume"})
)
job_df = (
    all_df[["jd_no", "job_text"]]
    .drop_duplicates(subset="jd_no")
    .rename(columns={"job_text": "text_job"})
)

resume_df.to_csv(os.path.join(OUTPUT_DIR, "all_resume.csv"), index=False)
job_df.to_csv(os.path.join(OUTPUT_DIR, "all_job.csv"), index=False)

print(f"Unique resumes: {len(resume_df):,}")
print(f"Unique jobs:    {len(job_df):,}")

Unique resumes: 643
Unique jobs:    351


In [12]:
# ── Save label files using pre-defined splits (no random splitting) ───
train_labels = train_df[["user_id", "jd_no", "satisfied"]]
valid_labels = valid_df[["user_id", "jd_no", "satisfied"]]
test_labels  = test_df[["user_id", "jd_no", "satisfied"]]

for name, df_labels in [("train_labeled_data", train_labels),
                         ("valid_classification_data", valid_labels),
                         ("test_classification_data", test_labels)]:
    path = os.path.join(OUTPUT_DIR, f"{name}.jsonl")
    df_labels.to_json(path, orient="records", lines=True)
    pos = (df_labels["satisfied"] == 1).sum()
    print(f"{name}: {len(df_labels):,} pairs ({pos:,} positive)")

train_labeled_data: 6,400 pairs (1,600 positive)
valid_classification_data: 800 pairs (200 positive)
test_classification_data: 800 pairs (200 positive)


In [13]:
# ── Build ranking evaluation files ─────────────────────────────────────
# rank_job.json: for each resume, rank candidate jobs
# rank_resume.json: for each job, rank candidate resumes

def build_rank_data(labels_df):
    """Build ranking dicts from label pairs."""
    rank_job = {}   # user_id -> {pos_jd_nos, neg_jd_nos}
    rank_resume = {}  # jd_no -> {pos_user_ids, neg_user_ids}

    for _, row in labels_df.iterrows():
        uid, jid, sat = row["user_id"], row["jd_no"], row["satisfied"]
        key = "pos" if sat == 1 else "neg"

        rank_job.setdefault(uid, {"pos": [], "neg": []})[key].append(jid)
        rank_resume.setdefault(jid, {"pos": [], "neg": []})[key].append(uid)

    # Keep only entries that have at least one positive
    rank_job = {k: v for k, v in rank_job.items() if len(v["pos"]) > 0 and len(v["neg"]) > 0}
    rank_resume = {k: v for k, v in rank_resume.items() if len(v["pos"]) > 0 and len(v["neg"]) > 0}
    return rank_job, rank_resume


rank_job, rank_resume = build_rank_data(test_labels)

with open(os.path.join(OUTPUT_DIR, "rank_job.json"), "w") as f:
    json.dump(rank_job, f)
with open(os.path.join(OUTPUT_DIR, "rank_resume.json"), "w") as f:
    json.dump(rank_resume, f)

print(f"rank_job entries:    {len(rank_job)} (resumes with both pos & neg jobs)")
print(f"rank_resume entries: {len(rank_resume)} (jobs with both pos & neg resumes)")

rank_job entries:    92 (resumes with both pos & neg jobs)
rank_resume entries: 64 (jobs with both pos & neg resumes)


In [14]:
# ── Also build valid ranking files ─────────────────────────────────────
valid_rank_job, valid_rank_resume = build_rank_data(valid_labels)

with open(os.path.join(OUTPUT_DIR, "valid_rank_job.json"), "w") as f:
    json.dump(valid_rank_job, f)
with open(os.path.join(OUTPUT_DIR, "valid_rank_resume.json"), "w") as f:
    json.dump(valid_rank_resume, f)

print(f"valid_rank_job:    {len(valid_rank_job)}")
print(f"valid_rank_resume: {len(valid_rank_resume)}")

valid_rank_job:    96
valid_rank_resume: 63


In [15]:
# ── BM25-compatible files ─────────────────────────────────────────────
# The BM25 tester expects a different ranking JSON format and trainable ID files.
BM25_DIR = os.path.join(OUTPUT_DIR, "bm25_format")
os.makedirs(BM25_DIR, exist_ok=True)

# 1. all_resume.csv and all_jd.csv — already generated, just copy/rename
import shutil
shutil.copy(os.path.join(OUTPUT_DIR, "all_resume.csv"), os.path.join(BM25_DIR, "all_resume.csv"))
shutil.copy(os.path.join(OUTPUT_DIR, "all_job.csv"), os.path.join(BM25_DIR, "all_jd.csv"))

# 2. trainable_resume.csv / trainable_jd.csv — IDs from training split
train_uids = train_labels["user_id"].drop_duplicates()
train_jids = train_labels["jd_no"].drop_duplicates()
train_uids.to_frame().to_csv(os.path.join(BM25_DIR, "trainable_resume.csv"), index=False)
train_jids.to_frame().to_csv(os.path.join(BM25_DIR, "trainable_jd.csv"), index=False)

# 3. Convert ranking JSONs from {pos/neg} format to BM25 format {jd_nos+labels}
def convert_rank_job_for_bm25(rank_job_dict):
    out = {}
    for uid, data in rank_job_dict.items():
        jd_nos = data["pos"] + data["neg"]
        labels = [1] * len(data["pos"]) + [0] * len(data["neg"])
        out[uid] = {"jd_nos": jd_nos, "labels": labels}
    return out

def convert_rank_resume_for_bm25(rank_resume_dict):
    out = {}
    for jid, data in rank_resume_dict.items():
        user_ids = data["pos"] + data["neg"]
        labels = [1] * len(data["pos"]) + [0] * len(data["neg"])
        out[jid] = {"user_ids": user_ids, "labels": labels}
    return out

bm25_rank_job = convert_rank_job_for_bm25(rank_job)
bm25_rank_resume = convert_rank_resume_for_bm25(rank_resume)

with open(os.path.join(BM25_DIR, "rank_job.json"), "w") as f:
    json.dump(bm25_rank_job, f)
with open(os.path.join(BM25_DIR, "rank_resume.json"), "w") as f:
    json.dump(bm25_rank_resume, f)

# ── MV-CoN / InEXIT / DPGNN compatible files ──────────────────────────
# Same as BM25 but uses 'satisfied' instead of 'labels' in ranking JSONs.
# Also needs the same train/valid/test JSONL files (already generated above).
MVCON_DIR = os.path.join(OUTPUT_DIR, "mvcon_format")
os.makedirs(MVCON_DIR, exist_ok=True)

shutil.copy(os.path.join(OUTPUT_DIR, "all_resume.csv"), os.path.join(MVCON_DIR, "all_resume.csv"))
shutil.copy(os.path.join(OUTPUT_DIR, "all_job.csv"), os.path.join(MVCON_DIR, "all_jd.csv"))
shutil.copy(os.path.join(OUTPUT_DIR, "train_labeled_data.jsonl"), os.path.join(MVCON_DIR, "train_labeled_data.jsonl"))
shutil.copy(os.path.join(OUTPUT_DIR, "valid_classification_data.jsonl"), os.path.join(MVCON_DIR, "valid_classification_data.jsonl"))
shutil.copy(os.path.join(OUTPUT_DIR, "test_classification_data.jsonl"), os.path.join(MVCON_DIR, "test_classification_data.jsonl"))

def convert_rank_job_for_mvcon(rank_job_dict):
    out = {}
    for uid, data in rank_job_dict.items():
        jd_nos = data["pos"] + data["neg"]
        satisfied = [1] * len(data["pos"]) + [0] * len(data["neg"])
        out[uid] = {"jd_nos": jd_nos, "satisfied": satisfied}
    return out

def convert_rank_resume_for_mvcon(rank_resume_dict):
    out = {}
    for jid, data in rank_resume_dict.items():
        user_ids = data["pos"] + data["neg"]
        satisfied = [1] * len(data["pos"]) + [0] * len(data["neg"])
        out[jid] = {"user_ids": user_ids, "satisfied": satisfied}
    return out

mvcon_rank_job = convert_rank_job_for_mvcon(rank_job)
mvcon_rank_resume = convert_rank_resume_for_mvcon(rank_resume)

with open(os.path.join(MVCON_DIR, "rank_job.json"), "w") as f:
    json.dump(mvcon_rank_job, f)
with open(os.path.join(MVCON_DIR, "rank_resume.json"), "w") as f:
    json.dump(mvcon_rank_resume, f)

print(f"\nBM25 files → {BM25_DIR}")
for f in sorted(os.listdir(BM25_DIR)):
    print(f"  {f}")
print(f"\nMV-CoN/InEXIT/DPGNN files → {MVCON_DIR}")
for f in sorted(os.listdir(MVCON_DIR)):
    print(f"  {f}")


BM25 files → /content/drive/MyDrive/resume_job_fit/bm25_format
  all_jd.csv
  all_resume.csv
  rank_job.json
  rank_resume.json
  trainable_jd.csv
  trainable_resume.csv

MV-CoN/InEXIT/DPGNN files → /content/drive/MyDrive/resume_job_fit/mvcon_format
  all_jd.csv
  all_resume.csv
  rank_job.json
  rank_resume.json
  test_classification_data.jsonl
  train_labeled_data.jsonl
  valid_classification_data.jsonl


In [16]:
# ── Summary of generated files ─────────────────────────────────────────
print("\nFiles in", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:45s} {size/1024:.1f} KB")


Files in /content/drive/MyDrive/resume_job_fit
  all_job.csv                                   951.0 KB
  all_resume.csv                                3802.6 KB
  bm25_format                                   4.0 KB
  converted_raw.json                            1035.5 KB
  mvcon_format                                  4.0 KB
  rank_job.json                                 30.4 KB
  rank_resume.json                              27.2 KB
  templates.json                                173.7 KB
  test_classification_data.jsonl                131.2 KB
  train_labeled_data.jsonl                      1050.0 KB
  valid_classification_data.jsonl               131.2 KB
  valid_rank_job.json                           34.2 KB
  valid_rank_resume.json                        27.9 KB


## 2. Create few-shot templates

We auto-build templates from "Good Fit" pairs in the training data.  
These are used as one-shot examples when prompting the LLM.

In [17]:
TEMPLATES_PATH = os.path.join(OUTPUT_DIR, "templates.json")
NUM_TEMPLATES = 20

# Pick Good Fit (label=1) pairs from training split as templates
good_fit_rows = train_df[train_df["satisfied"] == 1]
sampled = good_fit_rows.sample(n=min(NUM_TEMPLATES, len(good_fit_rows)), random_state=SEED)

templates = []
for _, row in sampled.iterrows():
    templates.append({
        "job": row["job_text"],
        "resume": row["resume_text"],
    })

with open(TEMPLATES_PATH, "w", encoding="utf-8") as f:
    json.dump(templates, f, indent=2, ensure_ascii=False)

print(f"Saved {len(templates)} templates from Good Fit pairs")
print(f"Job preview:    {templates[0]['job'][:200]}...")
print(f"Resume preview: {templates[0]['resume'][:200]}...")

Saved 20 templates from Good Fit pairs
Job preview:    This is a W2 contract in Minneapolis, MN 55402.Bank is looking for a successful Data Analyst to support our Enterprise Data Governance (EDG) and Data Privacy and Protection (DPP) initiatives. This rol...
Resume preview: Career OverviewExperienced Data Analyst with extensive
skill in Data Modelling, Analysis, and Reporting using SSMS, SSRS, Crystal

Reports,
and CRM applications. Detail-oriented professional with prof...


## 3. Generate hypothetical resumes via LLM

For each unique job, prompt Qwen2.5-32B-Instruct to generate an ideal resume.

In [13]:
# ── Load model ──────────────────────────────────────────────────────────
# Qwen2.5-32B-Instruct follows instructions directly without reasoning chains.
# 32B in bf16 ≈ 64GB — fits on a single A100 80GB.
MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Loaded {MODEL_ID}")

config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loaded deepseek-ai/DeepSeek-R1-Distill-Qwen-32B


In [14]:
def build_hyre_prompt(job_text: str, template: dict) -> str:
    """Construct the prompt exactly as the ConFit v2 codebase does."""
    return "\n".join([
        "Here is a template pair of matching resume and job:",
        "[The start of the example job]",
        template["job"],
        "[The end of the example job]",
        "[The start of the example resume]",
        template["resume"],
        "[The end of the example resume]",
        "You are a helpful assistant. Following the above example pair of job and resume, "
        "construct an ideal resume for the target job shown below. You should strictly follow "
        "the format of the given pairs, make sure the resume you give perfectly matches the "
        "target job, and directly return your answer in plain texts.",
        "[The start of the target job]",
        job_text,
        "[The end of the target job]",
    ])


def clean_llm_output(text: str) -> str:
    """Strip bracketed markers from the generated resume."""
    text = re.sub(r"\[The start of .*?\]\n?", "", text)
    text = re.sub(r"\n?\[The end of .*?\]?", "", text)
    return text.strip()


def generate_hyre(prompt: str, max_new_tokens: int = 2048) -> str:
    messages = [{"role": "user", "content": prompt}]
    chat_input = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    input_ids      = chat_input["input_ids"].to(llm_model.device)
    attention_mask = chat_input["attention_mask"].to(llm_model.device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    generated = output_ids[0, input_ids.shape[1]:]
    return llm_tokenizer.decode(generated, skip_special_tokens=True)


# Quick test with one job
sample_job = job_df.iloc[0]
prompt = build_hyre_prompt(sample_job["text_job"], random.choice(templates))
print("=== PROMPT (first 600 chars) ===")
print(prompt[:600])
print("...")

=== PROMPT (first 600 chars) ===
Here is a template pair of matching resume and job:
[The start of the example job]
LHH Recruitment Solutions is partnering with a company that will be relocating to St. Louis Park and looking for an experienced Accountant to support their centrally located clients in Food production. Here you will establish relationships within the home office as well as support manufacturing locations in month end procedure, journal entries as well as participate with the budgeting and forecasting processes.
Key Responsibilities: Perform a variety of accounting responsibilities as well as forecasting and budg
...


In [ ]:
# ── Generate for ALL unique jobs ───────────────────────────────────────
# Set MAX_JOBS to a small number to test first, then None for full run.
MAX_JOBS = None  # <-- set to None to process all jobs

jobs_to_process = job_df if MAX_JOBS is None else job_df.head(MAX_JOBS)

results = {}  # jd_no -> hypothetical resume text

# Resume from checkpoint if exists
checkpoint_path = os.path.join(OUTPUT_DIR, "converted_raw.json")
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        results = json.load(f)
    print(f"Resumed from checkpoint: {len(results)} already done")

for i, (_, row) in enumerate(tqdm(jobs_to_process.iterrows(), total=len(jobs_to_process), desc="Generating HYRE")):
    jd_no = row["jd_no"]
    if jd_no in results:
        continue  # skip already generated

    job_text = row["text_job"]
    template = random.choice(templates)
    prompt   = build_hyre_prompt(job_text, template)
    raw_output = generate_hyre(prompt)
    results[jd_no] = clean_llm_output(raw_output)

    # Checkpoint every 50 jobs
    if (i + 1) % 50 == 0:
        with open(checkpoint_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

# Final save
with open(checkpoint_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nGenerated {len(results)} hypothetical resumes")

Generating HYRE:   0%|          | 0/351 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

## 4. Merge & concatenate with original JD

In [ ]:
CONCAT_MODE = True  # recommended by the paper

job_text_lookup = dict(zip(job_df["jd_no"], job_df["text_job"]))

merged_rows = {"jd_no": [], "text_job": []}
for jd_no, hypo_resume in results.items():
    merged_rows["jd_no"].append(jd_no)
    if CONCAT_MODE:
        merged_rows["text_job"].append(
            job_text_lookup[jd_no] + "\n[Example resume]\n" + hypo_resume
        )
    else:
        merged_rows["text_job"].append(hypo_resume)

merged_df = pd.DataFrame(merged_rows)

suffix = "_concat" if CONCAT_MODE else ""
out_csv = os.path.join(OUTPUT_DIR, f"all_job_converted{suffix}.csv")
merged_df.to_csv(out_csv, index=False)
print(f"Saved merged job CSV ({len(merged_df)} rows) to {out_csv}")

Saved merged job CSV (5 rows) to /content/dataset/resume_job_fit/all_job_converted_concat.csv


## 5. Inspect the results

In [ ]:
idx = random.randint(0, len(merged_df) - 1)
sample = merged_df.iloc[idx]
jd_no = sample["jd_no"]

print("=" * 60)
print(f"JD_NO: {jd_no}")
print("=" * 60)

print("\n── ORIGINAL JOB ──")
print(job_text_lookup.get(jd_no, "(not found)")[:1000])

print("\n── HYPOTHETICAL RESUME (raw) ──")
print(results.get(jd_no, "(not found)")[:1000])

print("\n── FINAL text_job (what the encoder sees) ──")
print(sample["text_job"][:1500])

JD_NO: b5c4eb7fcb7f4f97162e050a6f3e4a8e6cc7781497858ca8445452604a036608

── ORIGINAL JOB ──
Net2Source Inc. is an award-winning total workforce solutions company recognized by Staffing Industry Analysts for our accelerated growth of 300% in the last 3 years with over 5500+ employees globally, with over 30+ locations in the US and global operations in 32 countries. We believe in providing staffing solutions to address the current talent gap  Right Talent  Right Time  Right Place  Right Price and acting as a Career Coach to our consultants.  
Role: Basel Business AnalystLocation: Washington, D.C.Work Mode: HybridHire Type: 6+ Month Contract (extendable)
JD: Role Specific Experience: 6+ years of relevant technical and business work experience The Candidates who have worked on Basel related projects in Credit risks or at least are aware of credit risk.  Banking & Financial domain experience, along with knowledge of risk management, familiarity with concepts of finance and accounting  Profi

In [ ]:
# ── HuggingFace login (only needed for gated models like Jina) ─────────
# login(token="hf_YOUR_TOKEN_HERE")

In [ ]:
# Token length distribution
enc_tokenizer = AutoTokenizer.from_pretrained("jinaai/jina-embeddings-v2-base-zh")

lengths = merged_df["text_job"].apply(lambda t: len(enc_tokenizer.encode(t)))
print(lengths.describe())
print(f"\nRows exceeding 3200 tokens: {(lengths > 3200).sum()} / {len(lengths)}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1994 > 512). Running this sequence through the model will result in indexing errors


count       5.000000
mean     1817.400000
std       373.019168
min      1236.000000
25%      1713.000000
50%      1919.000000
75%      1994.000000
max      2225.000000
Name: text_job, dtype: float64

Rows exceeding 3200 tokens: 0 / 5


## 6. Train ConFit v2

Free the LLM from GPU memory first, then run the training pipeline.

In [ ]:
import gc

del llm_model
del llm_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

GPU memory freed. Allocated: 65.5 GB


In [ ]:
import os
os.chdir(REPO_ROOT)
os.environ["PYTHONPATH"] = REPO_ROOT
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"Working directory: {os.getcwd()}")

Working directory: /content/ConFit-v2


In [ ]:
!python runners/trainer/train_confit.py \
  --save_path /content/drive/MyDrive/resume_job_fit/model_checkpoints/confit_v2_fit \
  --resume_data_path /content/drive/MyDrive/resume_job_fit/all_resume.csv \
  --job_data_path /content/drive/MyDrive/resume_job_fit/all_job_converted_concat.csv \
  --train_label_path /content/drive/MyDrive/resume_job_fit/train_labeled_data.jsonl \
  --valid_label_path /content/drive/MyDrive/resume_job_fit/valid_classification_data.jsonl \
  --classification_data_path /content/drive/MyDrive/resume_job_fit/test_classification_data.jsonl \
  --rank_resume_data_path /content/drive/MyDrive/resume_job_fit/rank_resume.json \
  --rank_job_data_path /content/drive/MyDrive/resume_job_fit/rank_job.json \
  --dataset_type recruiting_data_v2 \
  --num_resume_features 1 \
  --num_job_features 1 \
  --encode_all True \
  --embedding_method mean_pool \
  --pretrained_encoder jinaai/jina-embeddings-v2-base-zh \
  --do_normalize true \
  --temperature 0.05 \
  --do_both_rj_hard_neg true \
  --train_batch_size 4 \
  --val_batch_size 4 \
  --gradient_accumulation_steps 2 \
  --precision bf16 \
  --strategy deepspeed_stage_2 \
  --max_epochs 10

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_no

## 7. Mine hard negatives

Use the trained model to find "runner-up" negatives — resumes/jobs that score in the top 3-4% by cosine similarity but are not labeled positives. These make the second training pass much harder and more effective.

**Run this after Section 6 completes.**

In [ ]:
import glob, os

# ── Find the best checkpoint saved by Section 6 ────────────────────────
ckpt_dir = "/content/drive/MyDrive/resume_job_fit/model_checkpoints/confit_v2_fit"
ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "*.fp32"))) or \
        sorted(glob.glob(os.path.join(ckpt_dir, "*.ckpt")))

print("Available checkpoints:")
for c in ckpts:
    print(f"  {c}")

BEST_CKPT = ckpts[0] if ckpts else None
print(f"\nUsing: {BEST_CKPT}")

In [ ]:
# ── Mine hard negatives for the training set ───────────────────────────
!python src/utils/hard_negative_mining.py \
  --model_path {BEST_CKPT} \
  --file_path /content/drive/MyDrive/resume_job_fit/hard_negatives \
  --file_name hard_negatives_train.json \
  --resume_data_path /content/drive/MyDrive/resume_job_fit/all_resume.csv \
  --job_data_path /content/drive/MyDrive/resume_job_fit/all_job_converted_concat.csv \
  --classification_data_path /content/drive/MyDrive/resume_job_fit/train_labeled_data.jsonl \
  --dataset_type recruiting_data_v2 \
  --batch_size 16 \
  --seed 42 \
  --lower_bound 0.04 \
  --upper_bound 0.03 \
  --num_resume_features 1 \
  --num_job_features 1 \
  --encode_all True \
  --embedding_method mean_pool \
  --finegrained_loss noop \
  --pretrained_encoder jinaai/jina-embeddings-v2-base-zh \
  --do_normalize true \
  --temperature 0.05 \
  --do_both_rj_hard_neg true

In [ ]:
# ── Mine hard negatives for the validation set ─────────────────────────
!python src/utils/hard_negative_mining.py \
  --model_path {BEST_CKPT} \
  --file_path /content/drive/MyDrive/resume_job_fit/hard_negatives \
  --file_name hard_negatives_valid.json \
  --resume_data_path /content/drive/MyDrive/resume_job_fit/all_resume.csv \
  --job_data_path /content/drive/MyDrive/resume_job_fit/all_job_converted_concat.csv \
  --classification_data_path /content/drive/MyDrive/resume_job_fit/valid_classification_data.jsonl \
  --dataset_type recruiting_data_v2 \
  --batch_size 16 \
  --seed 42 \
  --lower_bound 0.04 \
  --upper_bound 0.03 \
  --num_resume_features 1 \
  --num_job_features 1 \
  --encode_all True \
  --embedding_method mean_pool \
  --finegrained_loss noop \
  --pretrained_encoder jinaai/jina-embeddings-v2-base-zh \
  --do_normalize true \
  --temperature 0.05 \
  --do_both_rj_hard_neg true

In [ ]:
# ── Verify hard negatives were saved to Drive ──────────────────────────
hn_dir = "/content/drive/MyDrive/resume_job_fit/hard_negatives"
for f in sorted(os.listdir(hn_dir)):
    size = os.path.getsize(os.path.join(hn_dir, f))
    print(f"  {f:45s} {size/1024:.1f} KB")

## 8. Re-train with hard negatives

Second training pass using the mined hard negatives. Model checkpoints saved to Drive.

In [ ]:
!python runners/trainer/train_confit.py \
  --save_path /content/drive/MyDrive/resume_job_fit/model_checkpoints/confit_v2_fit_hard_neg \
  --resume_data_path /content/drive/MyDrive/resume_job_fit/all_resume.csv \
  --job_data_path /content/drive/MyDrive/resume_job_fit/all_job_converted_concat.csv \
  --train_label_path /content/drive/MyDrive/resume_job_fit/train_labeled_data.jsonl \
  --valid_label_path /content/drive/MyDrive/resume_job_fit/valid_classification_data.jsonl \
  --classification_data_path /content/drive/MyDrive/resume_job_fit/test_classification_data.jsonl \
  --rank_resume_data_path /content/drive/MyDrive/resume_job_fit/rank_resume.json \
  --rank_job_data_path /content/drive/MyDrive/resume_job_fit/rank_job.json \
  --hard_negative_path /content/drive/MyDrive/resume_job_fit/hard_negatives/hard_negatives_train.json \
  --hard_negative_valid_path /content/drive/MyDrive/resume_job_fit/hard_negatives/hard_negatives_valid.json \
  --dataset_type recruiting_data_v2 \
  --num_resume_features 1 \
  --num_job_features 1 \
  --encode_all True \
  --embedding_method mean_pool \
  --pretrained_encoder jinaai/jina-embeddings-v2-base-zh \
  --do_normalize true \
  --temperature 0.05 \
  --do_both_rj_hard_neg true \
  --train_batch_size 4 \
  --val_batch_size 4 \
  --gradient_accumulation_steps 2 \
  --precision bf16 \
  --strategy deepspeed_stage_2 \
  --max_epochs 10

## 9. Evaluate BM25 baseline

Run BM25 on the test ranking data to get a baseline comparison for ConFit v2.

In [20]:
import sys
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from rank_bm25 import BM25L
from src.preprocess.word_tokenize import detect_language_and_word_tokenize
from src.evaluation.metrics import PrecomputedMetric
from src.evaluation.eval import EvalRanking

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Load data from BM25 format dir ────────────────────────────────────
BM25_DIR = os.path.join(OUTPUT_DIR, "bm25_format")

# Load data
all_resume_data = pd.read_csv(os.path.join(BM25_DIR, "all_resume.csv"))
all_resume_data["user_id"] = all_resume_data["user_id"].astype(str)
all_resume_data.index = all_resume_data["user_id"].values

all_job_data = pd.read_csv(os.path.join(BM25_DIR, "all_jd.csv"))
all_job_data["jd_no"] = all_job_data["jd_no"].astype(str)
all_job_data.index = all_job_data["jd_no"].values

rank_job_labels = json.load(open(os.path.join(BM25_DIR, "rank_job.json")))
rank_resume_labels = json.load(open(os.path.join(BM25_DIR, "rank_resume.json")))

print(f"Resumes: {len(all_resume_data)}, Jobs: {len(all_job_data)}")
print(f"Rank job entries: {len(rank_job_labels)}, Rank resume entries: {len(rank_resume_labels)}")

Resumes: 643, Jobs: 351
Rank job entries: 92, Rank resume entries: 64


In [21]:
# ── Tokenize all documents ────────────────────────────────────────────
def tokenize_text(text):
    """Simple word tokenization for single-field text."""
    if pd.isna(text):
        return []
    return detect_language_and_word_tokenize(str(text))

# Tokenize resumes
resume_id_to_words = {}
for _, row in tqdm(all_resume_data.iterrows(), total=len(all_resume_data), desc="Tokenizing resumes"):
    uid = str(row["user_id"])
    resume_id_to_words[uid] = tokenize_text(row["text_resume"])

# Tokenize jobs
job_id_to_words = {}
for _, row in tqdm(all_job_data.iterrows(), total=len(all_job_data), desc="Tokenizing jobs"):
    jid = str(row["jd_no"])
    job_id_to_words[jid] = tokenize_text(row["text_job"])

print(f"Tokenized {len(resume_id_to_words)} resumes, {len(job_id_to_words)} jobs")

Tokenizing resumes:   0%|          | 0/643 [00:00<?, ?it/s]

Tokenizing jobs:   0%|          | 0/351 [00:00<?, ?it/s]

Tokenized 643 resumes, 351 jobs


In [22]:
# ── Build BM25 indices and score all pairs ────────────────────────────
all_pairs_scored = {}

# Collect all docs needed for ranking
all_resume_docs, all_rid_to_idx = [], {}
all_job_docs, all_jid_to_idx = [], {}

for uid, data in rank_job_labels.items():
    if uid not in all_rid_to_idx and uid in resume_id_to_words:
        all_resume_docs.append(resume_id_to_words[uid])
        all_rid_to_idx[uid] = len(all_resume_docs) - 1
    for jid in data["jd_nos"]:
        if jid not in all_jid_to_idx and jid in job_id_to_words:
            all_job_docs.append(job_id_to_words[jid])
            all_jid_to_idx[jid] = len(all_job_docs) - 1

for jid, data in rank_resume_labels.items():
    if jid not in all_jid_to_idx and jid in job_id_to_words:
        all_job_docs.append(job_id_to_words[jid])
        all_jid_to_idx[jid] = len(all_job_docs) - 1
    for uid in data["user_ids"]:
        if uid not in all_rid_to_idx and uid in resume_id_to_words:
            all_resume_docs.append(resume_id_to_words[uid])
            all_rid_to_idx[uid] = len(all_resume_docs) - 1

# BM25 for ranking resumes (query = job, corpus = resumes)
bm25_resume = BM25L(all_resume_docs)
for jid, data in tqdm(rank_resume_labels.items(), desc="BM25 rank resume"):
    query = job_id_to_words.get(jid, [])
    rids = data["user_ids"]
    dids = [all_rid_to_idx[r] for r in rids if r in all_rid_to_idx]
    if not dids:
        continue
    scores = bm25_resume.get_batch_scores(query, dids)
    for s, rid in zip(scores, [r for r in rids if r in all_rid_to_idx]):
        all_pairs_scored[(str(rid), str(jid))] = s

# BM25 for ranking jobs (query = resume, corpus = jobs)
bm25_job = BM25L(all_job_docs)
for uid, data in tqdm(rank_job_labels.items(), desc="BM25 rank job"):
    query = resume_id_to_words.get(uid, [])
    jids = data["jd_nos"]
    dids = [all_jid_to_idx[j] for j in jids if j in all_jid_to_idx]
    if not dids:
        continue
    scores = bm25_job.get_batch_scores(query, dids)
    for s, jid in zip(scores, [j for j in jids if j in all_jid_to_idx]):
        all_pairs_scored[(str(uid), str(jid))] = s

print(f"Scored {len(all_pairs_scored):,} pairs")

BM25 rank resume:   0%|          | 0/64 [00:00<?, ?it/s]

BM25 rank job:   0%|          | 0/92 [00:00<?, ?it/s]

Scored 499 pairs


In [23]:
# ── Evaluate ranking metrics ──────────────────────────────────────────
metric = PrecomputedMetric(all_pairs_scored)

# Convert labels to EvalRanking format: {uid: {jd_nos:[], satisfied:[]}}
rank_job_eval = {}
for uid, data in rank_job_labels.items():
    rank_job_eval[uid] = {"jd_nos": data["jd_nos"], "satisfied": data["labels"]}

rank_resume_eval = {}
for jid, data in rank_resume_labels.items():
    rank_resume_eval[jid] = {"user_ids": data["user_ids"], "satisfied": data["labels"]}

rank_job_evaluator = EvalRanking(
    metric, test_rid_to_representation={}, test_jid_to_representation={},
    test_ranking_data=rank_job_eval, offline_mode=True
)
rank_resume_evaluator = EvalRanking(
    metric, test_rid_to_representation={}, test_jid_to_representation={},
    test_ranking_data=rank_resume_eval, offline_mode=True
)

rank_job_results, _ = rank_job_evaluator.evaluate()
rank_resume_results, _ = rank_resume_evaluator.evaluate()

print("\n=== BM25 Baseline Results ===")
print("\nRank Job (resume → find best jobs):")
for k, v in rank_job_results.items():
    print(f"  {k}: {v:.4f}")
print("\nRank Resume (job → find best resumes):")
for k, v in rank_resume_results.items():
    print(f"  {k}: {v:.4f}")


=== BM25 Baseline Results ===

Rank Job (resume → find best jobs):
  map: 0.7222
  ndcg: 0.8022
  ndcg@10: 0.8022
  recall@10: 1.0000

Rank Resume (job → find best resumes):
  map: 0.7250
  ndcg: 0.8161
  ndcg@10: 0.8147
  recall@10: 0.9969


## 10. Train & evaluate MV-CoN baseline

MV-CoN is a co-teaching network baseline. It requires training but uses a simpler architecture than ConFit.

In [ ]:
# ── Train MV-CoN ──────────────────────────────────────────────────────
os.chdir(REPO_ROOT)
os.environ["PYTHONPATH"] = REPO_ROOT
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!python runners/trainer/train_mvcon.py \
  --save_path /content/drive/MyDrive/resume_job_fit/model_checkpoints/mvcon \
  --resume_data_path /content/drive/MyDrive/resume_job_fit/all_resume.csv \
  --job_data_path /content/drive/MyDrive/resume_job_fit/all_job.csv \
  --train_label_path /content/drive/MyDrive/resume_job_fit/train_labeled_data.jsonl \
  --valid_label_path /content/drive/MyDrive/resume_job_fit/valid_classification_data.jsonl \
  --classification_data_path /content/drive/MyDrive/resume_job_fit/test_classification_data.jsonl \
  --rank_resume_data_path /content/drive/MyDrive/resume_job_fit/mvcon_format/rank_resume.json \
  --rank_job_data_path /content/drive/MyDrive/resume_job_fit/mvcon_format/rank_job.json \
  --dataset_type recruiting_data_v2 \
  --num_resume_features 1 \
  --num_job_features 1 \
  --pretrained_encoder jinaai/jina-embeddings-v2-base-zh \
  --train_batch_size 4 \
  --val_batch_size 4 \
  --gradient_accumulation_steps 4 \
  --weight_decay 1e-2 \
  --max_epochs 10 \
  --log_group resume_job_fit

## 11. Evaluate RawEmbed baseline (no training)

Encode all resumes and jobs with a pre-trained encoder (Jina v2) and score by cosine similarity. No training needed — this measures how well the raw encoder works out of the box.

In [24]:
import numpy as np
from src.model.modeling_bert import JinaBertModel
from src.model.configuration_bert import JinaBertConfig
from src.evaluation.eval import EvalRanking
from src.evaluation.metrics import PrecomputedMetric

# ── Load encoder (using local patched Jina model) ─────────────────────
ENCODER_ID = "jinaai/jina-embeddings-v2-base-zh"
enc_config = JinaBertConfig.from_pretrained(ENCODER_ID)
enc_config.emb_pooler = "mean"
enc_model = JinaBertModel.from_pretrained(ENCODER_ID, config=enc_config).cuda().eval()
print(f"Loaded {ENCODER_ID}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/322M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

Loaded jinaai/jina-embeddings-v2-base-zh


In [27]:
# ── Encode all resumes and jobs ────────────────────────────────────────
# Use the model's built-in encode() method for clean mean-pooled embeddings
resume_data = pd.read_csv(os.path.join(OUTPUT_DIR, "all_resume.csv"))
job_data = pd.read_csv(os.path.join(OUTPUT_DIR, "all_job.csv"))

resume_texts = resume_data["text_resume"].fillna("").tolist()
job_texts = job_data["text_job"].fillna("").tolist()

resume_embeddings = enc_model.encode(resume_texts, batch_size=16, normalize_embeddings=True, convert_to_numpy=True)
job_embeddings = enc_model.encode(job_texts, batch_size=16, normalize_embeddings=True, convert_to_numpy=True)

# Build ID → embedding maps
rid_to_emb = {str(uid): emb for uid, emb in zip(resume_data["user_id"], resume_embeddings)}
jid_to_emb = {str(jid): emb for jid, emb in zip(job_data["jd_no"], job_embeddings)}

print(f"Encoded {len(rid_to_emb)} resumes, {len(jid_to_emb)} jobs")

AttributeError: 'JinaBertModel' object has no attribute 'get_head_mask'

In [ ]:
# ── Score all test pairs by cosine similarity ──────────────────────────
# Load ranking labels (convert pos/neg to jd_nos/satisfied format for EvalRanking)
rank_job_raw = json.load(open(os.path.join(OUTPUT_DIR, "rank_job.json")))
rank_resume_raw = json.load(open(os.path.join(OUTPUT_DIR, "rank_resume.json")))

# Convert to EvalRanking format
rank_job_data = {}
for uid, data in rank_job_raw.items():
    rank_job_data[uid] = {"jd_nos": data["pos"] + data["neg"], "satisfied": [1]*len(data["pos"]) + [0]*len(data["neg"])}
rank_resume_data = {}
for jid, data in rank_resume_raw.items():
    rank_resume_data[jid] = {"user_ids": data["pos"] + data["neg"], "satisfied": [1]*len(data["pos"]) + [0]*len(data["neg"])}

all_pairs_scored = {}
for uid, data in rank_job_data.items():
    if uid not in rid_to_emb:
        continue
    r_emb = rid_to_emb[uid]
    for jid in data["jd_nos"]:
        if jid in jid_to_emb:
            score = float(np.dot(r_emb, jid_to_emb[jid]))
            all_pairs_scored[(uid, jid)] = score

for jid, data in rank_resume_data.items():
    if jid not in jid_to_emb:
        continue
    j_emb = jid_to_emb[jid]
    for uid in data["user_ids"]:
        if uid in rid_to_emb:
            score = float(np.dot(rid_to_emb[uid], j_emb))
            all_pairs_scored[(uid, jid)] = score

# Evaluate
metric = PrecomputedMetric(all_pairs_scored)

rank_job_evaluator = EvalRanking(
    metric, test_rid_to_representation={}, test_jid_to_representation={},
    test_ranking_data=rank_job_data, offline_mode=True
)
rank_resume_evaluator = EvalRanking(
    metric, test_rid_to_representation={}, test_jid_to_representation={},
    test_ranking_data=rank_resume_data, offline_mode=True
)

rank_job_results, _ = rank_job_evaluator.evaluate()
rank_resume_results, _ = rank_resume_evaluator.evaluate()

print(f"\n=== RawEmbed Baseline ({ENCODER_ID}) ===")
print("\nRank Job (resume → find best jobs):")
for k, v in rank_job_results.items():
    print(f"  {k}: {v:.4f}")
print("\nRank Resume (job → find best resumes):")
for k, v in rank_resume_results.items():
    print(f"  {k}: {v:.4f}")

# Free encoder
del enc_model
torch.cuda.empty_cache()

## 12. Train & evaluate InEXIT baseline

InEXIT uses hierarchical attention to model interactions between resume/job text fields. Requires training.

In [ ]:
# ── Train InEXIT ──────────────────────────────────────────────────────
os.chdir(REPO_ROOT)
os.environ["PYTHONPATH"] = REPO_ROOT
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!python runners/trainer/train_inexit.py \
  --save_path /content/drive/MyDrive/resume_job_fit/model_checkpoints/inexit \
  --resume_data_path /content/drive/MyDrive/resume_job_fit/all_resume.csv \
  --job_data_path /content/drive/MyDrive/resume_job_fit/all_job.csv \
  --train_label_path /content/drive/MyDrive/resume_job_fit/train_labeled_data.jsonl \
  --valid_label_path /content/drive/MyDrive/resume_job_fit/valid_classification_data.jsonl \
  --classification_data_path /content/drive/MyDrive/resume_job_fit/test_classification_data.jsonl \
  --rank_resume_data_path /content/drive/MyDrive/resume_job_fit/mvcon_format/rank_resume.json \
  --rank_job_data_path /content/drive/MyDrive/resume_job_fit/mvcon_format/rank_job.json \
  --dataset_type recruiting_data_v2 \
  --num_resume_features 1 \
  --num_job_features 1 \
  --pretrained_encoder jinaai/jina-embeddings-v2-base-zh \
  --train_batch_size 8 \
  --val_batch_size 8 \
  --gradient_accumulation_steps 2 \
  --weight_decay 1e-2 \
  --max_epochs 10 \
  --log_group resume_job_fit